In [1]:
#!/usr/bin/env python3
"""
LLM部署系统测试脚本

该脚本用于在远程测试环境中对LLM部署系统进行系统性测试，包括：
1. 环境检查
2. 模型加载/卸载测试
3. 基础功能测试
4. 专用模型类型测试
5. 性能基准测试
6. 测试报告生成

注意：该脚本仅设计为执行测试准备和测试流程，不得在当前环境实际运行。
"""

import argparse
import os
import sys
import json
import time
import requests
import psutil
import logging
from datetime import datetime
from typing import Dict, List, Tuple, Optional


# 安全的API请求函数
def safe_api_request(method, url, **kwargs):
    """安全的API请求函数，处理各种网络错误"""
    try:
        if method.lower() == 'get':
            response = requests.get(url, **kwargs)
        elif method.lower() == 'post':
            response = requests.post(url, **kwargs)
        else:
            raise ValueError(f"不支持的HTTP方法: {method}")
        return response
    except requests.exceptions.Timeout:
        print(f"API请求超时: {url}")
        return None
    except requests.exceptions.ConnectionError:
        print(f"API连接错误: {url}")
        return None
    except requests.exceptions.RequestException as e:
        print(f"API请求错误: {url}, 错误: {str(e)}")
        return None
    except Exception as e:
        print(f"未知错误: {url}, 错误: {str(e)}")
        return None

# 全局配置
DEFAULT_API_URL = "http://127.0.0.1:8000"
MODEL_WEIGHTS_DIR = os.environ.get('MODEL_WEIGHTS_DIR', '/data2/home/songzhihua/LLM/llm_playground/model_weights')

# 测试模型列表
TEST_MODELS = {
    "text_models": [
        "Qwen/Qwen3-8B",
        "Qwen/Qwen3-30B-A3B-Instruct-2507"
    ],
    "multimodal_models": [
        "Qwen/Qwen3-VL-8B-Thinking"
    ],
    "embedding_models": [
        "BAAI/bge-large-zh-v1.5",
        "BAAI/bge-m3"
    ]
}


In [19]:
# api_url = DEFAULT_API_URL
api_url = 'http://127.0.0.1:8080'

In [41]:
# 获取所有可用的模型列表
response = safe_api_request('get', f"{api_url}/model/list", timeout=10)
response.json()

{}

In [39]:
response

<Response [200]>

In [40]:
for model_name in response.json():
    response = safe_api_request(
        'post',
        f"{api_url}/model/unload",
        json={"model_name": model_name},
        timeout=60
    )
    

In [31]:
# 加载指定模型
model_name = 'Qwen/Qwen3-8B'
# model_name = 'BAAI/bge-m3'
# model_name = 'Qwen/Qwen3-VL-8B-Thinking'
response = safe_api_request(
    'post',
    f"{api_url}/model/load",
    json={"model_name": model_name},
    timeout=300  # 给大型模型足够的加载时间
)
response.json()

{'status': 'loaded'}

In [17]:
# 卸载指定模型
model_name = 'Qwen/Qwen3-8B'
# model_name = 'BAAI/bge-large-zh-v1.5'
response = safe_api_request(
    'post',
    f"{api_url}/model/unload",
    json={"model_name": model_name},
    timeout=60
)
response.json()

{'status': 'unloaded',
 'message': '模型 Qwen/Qwen3-8B 已卸载',
 'cleanup_result': {'cleaned_count': 0, 'cleaned_pids': []}}

In [ ]:
cmd_load = """
curl -X POST http://127.0.0.1:8080/model/load   -H "Content-Type: application/json"   -d '{
    "model_name": "BAAI/bge-m3"
}'

curl -X POST http://127.0.0.1:8080/model/load   -H "Content-Type: application/json"   -d '{
    "model_name": "Qwen/Qwen3-8B"
}'

curl -X POST http://127.0.0.1:8080/model/load   -H "Content-Type: application/json"   -d '{
    "model_name": "BAAI/bge-large-zh-v1.5"
}'
"""

cmd_unload = """
curl -X POST http://127.0.0.1:8080/model/unload   -H "Content-Type: application/json"   -d '{
    "model_name": "BAAI/bge-m3"
}'

curl -X POST http://127.0.0.1:8080/model/unload   -H "Content-Type: application/json"   -d '{
    "model_name": "Qwen/Qwen3-8B"
}'

curl -X POST http://127.0.0.1:8080/model/unload   -H "Content-Type: application/json"   -d '{
    "model_name": "Qwen/Qwen3-VL-8B-Thinking"
}'
"""

cmd_cleanup = """
curl -X POST http://127.0.0.1:8080/model/cleanup
"""

cmd_generate = """
curl -X POST "http://127.0.0.1:8080/generate" \
-H "Content-Type: application/json" \
-d '{
  "model_name": "Qwen/Qwen3-8B",
  "messages": [
    {
      "role": "user",
      "content": "直接返回True，不得返回任何多余内容"
    }
  ],
  "max_new_tokens": 1024
}'


curl -X POST "http://127.0.0.1:8080/generate" \
-H "Content-Type: application/json" \
-d '{
  "model_name": "Qwen/Qwen3-8B",
  "messages": [
    {
      "role": "user",
      "content": "什么是人工智能"
    }
  ],
  "max_new_tokens": 1024
}'
"""

In [36]:
# 清理僵尸进程
response = safe_api_request('post', f"{api_url}/model/cleanup", timeout=10)
response.json()

{'cleaned_count': 0, 'cleaned_pids': []}

In [ ]:
# 测试模型生成 
model_name = 'Qwen/Qwen3-VL-8B-Thinking'
response = safe_api_request(
    'post',
    f"{api_url}/generate",
    json={
        "model_name": model_name,
        "messages": [{"role": "user", "content": "如果x=5，y=3，那么x²+y³等于多少？请详细思考过程。"}],
        "max_new_tokens": 1024
    },
    timeout=300
)
response.json()

In [35]:
model_name = 'BAAI/bge-large-zh-v1.5'
test_texts = ["这是一个测试句子", "这是另一个测试句子", "这是第三个测试句子"]

# 修复：使用safe_api_request替代直接requests调用
response = safe_api_request(
    'post',
    f"{api_url}/embedding",
    json={
        "model_name": model_name,
        "texts": test_texts,
        "query": False
    },
    timeout=300
)

response.json()

{'embedding': [[0.01081085205078125,
   -0.011138916015625,
   -0.0158843994140625,
   0.017578125,
   0.040496826171875,
   0.01068878173828125,
   0.00522613525390625,
   -0.0261383056640625,
   -0.039764404296875,
   0.03167724609375,
   -0.01317596435546875,
   -0.0013341903686523438,
   -0.0241851806640625,
   -0.029632568359375,
   0.021270751953125,
   0.0300445556640625,
   0.01715087890625,
   -0.021759033203125,
   0.045654296875,
   -0.017608642578125,
   0.060546875,
   -0.03167724609375,
   -0.05206298828125,
   -0.0513916015625,
   0.04913330078125,
   -0.007659912109375,
   -0.0462646484375,
   -0.01873779296875,
   0.0067291259765625,
   0.0173492431640625,
   -0.005702972412109375,
   0.021392822265625,
   -0.0005764961242675781,
   -0.011322021484375,
   -0.0027446746826171875,
   0.0008931159973144531,
   -0.043426513671875,
   -0.0186614990234375,
   -0.01044464111328125,
   -0.00033211708068847656,
   -0.0180206298828125,
   -0.0072784423828125,
   -0.0004682540893

In [ ]:
model_name = 'Qwen/Qwen3-VL-8B-Thinking'
response = safe_api_request(
    'post',
    f"{api_url}/generate",
    json={
        "model_name": model_name,
        "messages": [
            {
                "role": "user", "content": [
                    {
                        "type": "image",
                        "image": "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg",
                    },
                    {
                        "type": "text", 
                        "text": "Describe this image."
                    },
                ]
            }
        ],
        "max_new_tokens": 1024
    },
    timeout=300
)
print(response.status_code)
print(response.text) # 依然先打印 text

500
Internal Server Error


In [ ]:
response.json()

In [37]:
model_name = 'Qwen/Qwen3-8B'
response = safe_api_request(
    'post',
    f"{api_url}/generate",
    json={
        "model_name": model_name,
        "messages": [{"role": "user", "content": "什么是人工智能？"}],
        "max_new_tokens": 1024
    },
    timeout=300
)
print(response.status_code)
print(response.text) # 依然先打印 text

200
{"thinking":[""],"response":["人工智能（Artificial Intelligence，简称AI）是指由人创造的能够执行通常需要人类智能才能完成的复杂任务的系统或机器。它是一门融合了计算机科学、数学、心理学、语言学、哲学等多个学科的交叉学科。\n\n### 人工智能的核心目标：\n人工智能的目标是让机器具备类似人类的智能行为，例如：\n- 学习（从数据中提取知识）\n- 推理（逻辑推理、演绎、归纳）\n- 问题解决\n- 语言理解\n- 感知（视觉、听觉等）\n- 自主决策\n\n---\n\n### 人工智能的分类：\n\n根据能力的强弱，人工智能可以分为以下几类：\n\n#### 1. **弱人工智能（Narrow AI）**\n- 专注于执行特定任务，目前大多数AI应用都属于这一类。\n- 例如：\n  - 虚拟助手（如Siri、小爱同学）\n  - 推荐系统（如淘宝、抖音推荐）\n  - 图像识别（如人脸识别、车牌识别）\n  - 自动驾驶汽车（如特斯拉自动驾驶）\n  - 语音识别（如语音转文字）\n\n#### 2. **强人工智能（General AI）**\n- 指具有与人类相当甚至超越人类的通用智能，能够理解、学习和执行任何智力任务。\n- 目前尚未实现，仍属于理论研究阶段。\n\n#### 3. **超级人工智能（Superintelligent AI）**\n- 指智能水平远超人类的AI，能够解决几乎所有人类无法解决的问题。\n- 这是未来可能的发展方向，但目前仍属于科幻或理论设想。\n\n---\n\n### 人工智能的基本技术：\n\n1. **机器学习（Machine Learning）**  \n   - 通过数据训练模型，使系统能够自动学习并做出预测或决策。\n   - 常见类型包括：监督学习、无监督学习、强化学习等。\n\n2. **深度学习（Deep Learning）**  \n   - 基于神经网络的机器学习方法，特别适合处理图像、语音、文本等复杂数据。\n   - 是当前AI技术中最强大的方法之一。\n\n3. **自然语言处理（NLP）**  \n   - 使计算机能够理解、生成和处理人类语言。\n   - 应用包括机器翻译、聊天机器人、情感分析等。\n\n4. **计算机视觉（Com

In [ ]:
model_name = 'Qwen/Qwen3-8B'
response = safe_api_request(
    'post',
    f"{api_url}/generate",
    json={
        "model_name": model_name,
        "messages": [{"role": "user", "content": "直接返回True，不得返回任何多余内容"}],
        "max_new_tokens": 1024
    },
    timeout=300
)
print(response.status_code)
print(response.text) # 依然先打印 text

200
{"thinking":[""],"response":["True"]}


In [ ]:
from models import *

In [ ]:
import os 
dir_path = '/data2/home/songzhihua/LLM/llm_playground/model_weights'
model_name = 'Qwen/Qwen3-8B'
model, processor = load_model_and_processor(
    model_name,
    os.path.join(dir_path, model_name)
)

In [ ]:
respond = generate_response(
    model, 
    processor, 
    messages=[{"role": "user", "content": "什么是人工智能？"}], 
    enable_thinking = ENABLE_THINKING, 
    max_new_tokens = MAX_NEW_TOKENS,
)

In [ ]:
respond